# Report statistics

Every number and figure in the report comes from this notebook. It reads saved result files from `results/`
(copied from Data Lab by `scripts/collect_results.py`) and the region config. It does not train, and no result is typed in by hand.
A missing file stops the notebook with a message that says which step to run.

Order of work: train and evaluate (`run_all_datalab.ipynb`, `multi_year.ipynb`), then `python scripts/collect_results.py`, then run this notebook top to bottom and commit it with its outputs.
Tables go to `results/tables/`, figures to `docs/figures/`, and a summary of all numbers to `results/report_numbers.md`.

In [ ]:
import os, json, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
RES, FIG = ROOT / "results", ROOT / "docs" / "figures"
TAB = RES / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)

SHIPPED = "V12_window3_balanced_stats"     # the model of the report
BOUNDARY = "V16_window3_boundary"          # the boundary-loss variant (reported as an ablation)
R21 = RES / "runs"                         # models trained on 2021 only
RMY = RES / "runs_multiyear"               # models trained on 2021-2024 (the shipped ensemble)
DATA = Path(os.environ.get("LULC_DATA", "/home/jovyan/vault/lulc/raw"))   # only needed for the class distribution and the orbit check

plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
NUMBERS = []                                # (section, text) lines for results/report_numbers.md

def need(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing. Run the training/evaluation notebooks, then scripts/collect_results.py, then this notebook.")
    return path

def jload(path): return json.loads(need(path).read_text())
def table(df, name, note=""):
    df.to_csv(TAB / f"{name}.csv", index=False)
    NUMBERS.append((name, note, df))
    return df

print("root:", ROOT)

## 1. Regions and splits

In [ ]:
from lulc.config import load_config
label_year, regions = load_config(ROOT / "configs" / "regions.yaml")
rdf = pd.DataFrame([{"region": r.key, "split": r.split, "type": "city" if r.half_size_km > 6 else "rural", "side_km": 2 * r.half_size_km, "climate": r.climate} for r in regions])
splits = rdf.groupby(["split", "type"]).size().unstack(fill_value=0)
splits["total"] = splits.sum(axis=1)
splits = splits.reset_index()
display(table(splits, "splits", "Regions per split. Test regions are never used for training or model selection."))
print("Regions in total:", len(rdf), "| label year in config:", label_year)

## 2. Class distribution of the training labels

In [ ]:
from lulc.classes import CLASS_NAMES, N_CLASSES, remap_dynamic_world
cache = TAB / "class_distribution.csv"
if DATA.exists():
    counts = np.zeros(N_CLASSES)
    train_keys = [r.key for r in regions if r.split == "train"]
    n_files = 0
    for k in train_keys:
        for m in (1, 4, 7, 10):                     # four months spread over the year are enough for a share
            f = DATA / k / f"{label_year}-{m:02d}.npz"
            if f.exists():
                with np.load(f) as z:
                    lab = remap_dynamic_world(z["dw_label"])
                counts += np.bincount(lab[lab < N_CLASSES].ravel(), minlength=N_CLASSES)[:N_CLASSES]
                n_files += 1
    dist = pd.DataFrame({"class": CLASS_NAMES, "share_pct": 100 * counts / counts.sum()}).sort_values("share_pct", ascending=False)
    dist.to_csv(cache, index=False)
    print(f"Computed from {n_files} month files of {len(train_keys)} training regions (months 1, 4, 7, 10 of {label_year}); ignored pixels excluded.")
else:
    dist = pd.read_csv(need(cache))
    print("Data folder not found; using the saved table from the last run on Data Lab.")
dist["share_pct"] = dist["share_pct"].round(1)
display(table(dist, "class_distribution", "Share of labelled pixels in the training regions."))

## 3. Orbit direction check

In [ ]:
saved = RES / "orbit_check.txt"
if DATA.exists():
    out = subprocess.run([sys.executable, str(ROOT / "scripts" / "check_orbit.py"), "--data", str(DATA)], capture_output=True, text=True, cwd=ROOT)
    saved.write_text(out.stdout + out.stderr)
print(need(saved).read_text()[-2500:])

## 4. Variant comparison on the validation regions

In [ ]:
rows = []
for root, tag in ((R21, "trained on 2021"), (RMY, "trained on 2021-2024")):
    for p in sorted(root.glob("*/seed*/summary.json")):
        j = json.loads(p.read_text()); rows.append({**j, "trained": tag})
need(R21)
sm = pd.DataFrame(rows)
var = sm.groupby(["trained", "variant"]).best_val_miou.agg(mean="mean", std="std", seeds="count").round(3).reset_index()
display(table(var, "variant_table", "Best validation mIoU per variant, mean and standard deviation over seeds. Differences below about 0.05 are seed noise."))
print("Standard deviation range:", var["std"].min(), "to", var["std"].max(), "| median", round(var["std"].median(), 3))

## 5. Training curves (Figure 1)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
styles = {0: "-", 1: "--", 2: ":"}
hist_files = sorted((RMY / SHIPPED).glob("seed*/history.csv"))
if not hist_files:
    raise FileNotFoundError(f"No history.csv for {SHIPPED} in {RMY}. Run scripts/collect_results.py.")
for p in hist_files:
    s = int(p.parent.name.replace("seed", "")); h = pd.read_csv(p)
    ax[0].plot(h.epoch, h.train_loss, styles.get(s, "-"), color="0.2", label=f"seed {s}")
    ax[1].plot(h.epoch, h.val_miou, styles.get(s, "-"), color="0.2", label=f"seed {s}")
ax[0].set(xlabel="epoch", ylabel="training loss", title="Training loss")
ax[1].set(xlabel="epoch", ylabel="validation mIoU", title="Validation mIoU (2024 validation regions)")
ax[1].legend(frameon=False)
fig.tight_layout(); fig.savefig(FIG / "fig1_training_curves.png", dpi=200); plt.show()
best = pd.DataFrame([{"seed": p.parent.name, "best_epoch": int(pd.read_csv(p).val_miou.idxmax()) + 1, "best_val_miou": round(pd.read_csv(p).val_miou.max(), 3), "epochs_run": len(pd.read_csv(p))} for p in hist_files])
display(table(best, "training_summary", "Best epoch per seed of the shipped model."))

## 6. Test on 2021: models trained on 2021, regions never seen

In [ ]:
def eval_dir(runs, variant, tag=""): return need(runs / f"eval_{variant}{tag}")
d21 = eval_dir(R21, SHIPPED)
m21 = jload(d21 / "metrics.json")
def score_rows(m, keys):
    return pd.DataFrame([{"set": k, "mIoU": round(m[k]["miou"], 3), "ci_low": round(m[k]["miou_lo"], 3), "ci_high": round(m[k]["miou_hi"], 3),
                          "OA": round(m[k]["oa"], 3), "regions": m[k]["n_cities"]} for k in keys if k in m])
sets = ["dw", "dw_cities", "dw_rural", "esri", "wc", "dw_wc_agree", "agree"]
labels = {"dw": "vs Dynamic World (all)", "dw_cities": "vs Dynamic World, cities", "dw_rural": "vs Dynamic World, rural", "esri": "vs Esri 2021 (independent)",
          "wc": "vs WorldCover 2021 (independent)", "dw_wc_agree": "vs Dynamic World, only where WorldCover agrees", "agree": "vs pixels WorldCover and Esri call the same"}
t21 = score_rows(m21, sets); t21["set"] = t21["set"].map(labels)
display(table(t21, "test_2021", "Test 2021. The row 'only where WorldCover agrees' is the easier score some projects report; the main score is the first row."))
pc = pd.read_csv(need(d21 / "per_class.csv"))
pcd = pc[pc.set == "dw"][["class", "iou", "f1", "precision", "recall", "support_px"]].round(3)
display(table(pcd, "per_class_2021", "Per-class scores, test 2021, all regions pooled, vs Dynamic World."))

### Confusion matrix (Figure 2)

In [ ]:
cm = np.array(m21["dw"]["confusion"], float)
norm = cm / cm.sum(axis=1, keepdims=True).clip(1)
fig, ax = plt.subplots(figsize=(4.6, 4))
im = ax.imshow(norm, cmap="Greys", vmin=0, vmax=1)
ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha="right"); ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        ax.text(j, i, f"{norm[i, j]:.2f}", ha="center", va="center", color="white" if norm[i, j] > 0.5 else "black", fontsize=7)
ax.set(xlabel="predicted", ylabel="Dynamic World label", title="Confusion matrix, test 2021 (row-normalised)")
fig.tight_layout(); fig.savefig(FIG / "fig2_confusion_matrix.png", dpi=200); plt.show()

## 7. Worst and best regions, and the area warning

In [ ]:
from scipy.stats import spearmanr
pcity = pd.read_csv(need(d21 / "per_city.csv"))
pcity = pcity.merge(rdf[["region", "type"]], on="region", how="left")
worst = pcity.sort_values("dw_miou")[["region", "type", "climate", "dw_miou", "dw_oa", "n_major_classes", "mean_conf"]].round(3)
display(table(worst.head(8), "worst_regions_2021", "Lowest-scoring test regions in 2021. One-class regions are flagged by n_major_classes."))
rho = spearmanr(pcity.mean_conf, pcity.dw_miou)[0]
print(f"Spearman(mean confidence, mIoU) over {len(pcity)} regions: {rho:.2f}")
NUMBERS.append(("spearman_2021", f"Rank correlation between mean confidence and mIoU, 2021: {rho:.2f}", None))

## 8. Test on 2025 and 2026: unseen regions in unseen years

In [ ]:
rows, cities = [], {}
for y in (2025, 2026):
    for name, runs, var, tag in [("trained on 2021 only", R21, SHIPPED, f"_y{y}_shipped"),
                                 ("trained on 2021-2024 (shipped)", RMY, SHIPPED, f"_y{y}"),
                                 ("trained on 2021-2024, boundary loss", RMY, BOUNDARY, f"_y{y}")]:
        d = runs / f"eval_{var}{tag}"
        if not d.exists():
            print("missing (skipped):", d.name); continue
        m = jload(d / "metrics.json")
        for k, lab in (("dw", "all"), ("dw_cities", "cities"), ("dw_rural", "rural")):
            if k in m:
                rows.append({"year": y, "model": name, "regions": lab, "mIoU": round(m[k]["miou"], 3), "ci_low": round(m[k]["miou_lo"], 3),
                             "ci_high": round(m[k]["miou_hi"], 3), "OA": round(m[k]["oa"], 3), "n": m[k]["n_cities"]})
        if "shipped" not in name and var == SHIPPED and "boundary" not in name:
            cities[y] = pd.read_csv(d / "per_city.csv")
t2526 = pd.DataFrame(rows)
if t2526.empty:
    raise FileNotFoundError("No 2025/2026 evaluation folders found in results/. Run the evaluation cell of multi_year.ipynb and collect_results.py.")
display(table(t2526, "test_2025_2026", "Pooled mIoU against Dynamic World of the same year. No independent map exists for these years."))

In [ ]:
for y, c in cities.items():
    c = c.merge(rdf[["region", "type"]], on="region", how="left")
    multi = c[c.n_major_classes >= 2]
    rho = spearmanr(c.mean_conf, c.dw_miou)[0]
    print(f"{y}: worst regions: " + ", ".join(f"{r.region} {r.dw_miou:.3f}" for r in c.sort_values('dw_miou').head(5).itertuples()))
    print(f"{y}: Spearman(mean confidence, mIoU) {rho:.2f} over {len(c)} regions; regions with one major class: {(c.n_major_classes < 2).sum()}")

## 9. Class borders

In [ ]:
rows = []
for y in (2025, 2026):
    for name, runs, var, tag in [("trained on 2021 only", R21, SHIPPED, f"_y{y}_shipped"), ("trained on 2021-2024 (shipped)", RMY, SHIPPED, f"_y{y}"),
                                 ("trained on 2021-2024, boundary loss", RMY, BOUNDARY, f"_y{y}")]:
        d = runs / f"eval_{var}{tag}"
        if not (d / "boundary.csv").exists():
            continue
        pooled = d / "boundary_pooled.json"
        if pooled.exists():
            j = json.loads(pooled.read_text()); rows.append({"year": y, "model": name, "boundary_F1_raw": round(j["bf1_raw"], 3), "boundary_F1_majority3": round(j["bf1_majority"], 3),
                                                              "pred_over_ref_border_px": round(j["pred_over_ref_border_px"], 2), "how": "pooled"})
        else:
            b = pd.read_csv(d / "boundary.csv")
            rows.append({"year": y, "model": name, "boundary_F1_raw": round(b.bf1_raw.mean(), 3), "boundary_F1_majority3": round(b.bf1_majority.mean(), 3),
                         "pred_over_ref_border_px": round(b.pred_border_px.sum() / max(b.ref_border_px.sum(), 1), 2), "how": "mean of regions (re-run evaluate.py for the pooled value)"})
bt = pd.DataFrame(rows)
display(table(bt, "boundary_f1", "Boundary F1 against Dynamic World, 1 pixel tolerance."))

## 10. False change on stable land

In [ ]:
ch = pd.read_csv(need(d21 / "change.csv"))
ch = ch[ch.maps != "member_spread_pp_mean"]
fc = ch.groupby("maps")[["flip_rate_pct", "monthly_churn_pp", "net_total_change_pp"]].mean().round(2).reset_index()
display(table(fc, "false_change", f"Change on pixels both annual maps call stable, 2021, mean over {ch.region.nunique()} test regions."))
spread = pd.read_csv(d21 / "change.csv")
spread = spread[spread.maps == "member_spread_pp_mean"]
if len(spread):
    print("Mean seed spread of class shares (pp):", round(spread.monthly_churn_pp.mean(), 2))

## 11. Score by calendar month

In [ ]:
bm = pd.read_csv(need(d21 / "by_month.csv"), index_col=0).round(3).reset_index().rename(columns={"index": "month"})
display(table(bm[["month", "OA", "mIoU"]], "by_month_2021", "Pooled score per calendar month, test 2021."))
fig, ax = plt.subplots(figsize=(4.6, 2.6))
ax.plot(bm.month, bm.mIoU, "-o", color="0.2", ms=3); ax.set(xlabel="month", ylabel="mIoU", title="Test 2021 by month", xticks=range(1, 13))
fig.tight_layout(); fig.savefig(FIG / "fig4_by_month.png", dpi=200); plt.show()

## 12. Case study: Heidelberg, 2021

Shows the saved check against the independent maps. The table is produced by the prototype's independent-map check; save its output as `results/heidelberg_2021_independent_check.csv` (columns: reference, agreement_pct, miou).

In [ ]:
hp = RES / "heidelberg_2021_independent_check.csv"
if hp.exists():
    display(table(pd.read_csv(hp), "heidelberg_2021", "Case study Heidelberg 2021 against Esri and WorldCover."))
else:
    print("Not found:", hp, "- save the prototype's comparison table there, then re-run this cell.")
from IPython.display import Image
fig3 = FIG / "heidelberg_2021_independent_check.png"
if fig3.exists():
    display(Image(str(fig3)))

## 13. Write all numbers to one file

In [ ]:
def to_md(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "| " + " | ".join("---" for _ in cols) + " |"]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)
out = ["# Report numbers (generated by notebooks/report_stats.ipynb, do not edit by hand)\n"]
for name, note, df in NUMBERS:
    out.append(f"## {name}\n\n{note}\n")
    if df is not None:
        out.append(to_md(df) + "\n")
(RES / "report_numbers.md").write_text("\n".join(out))
print("Wrote", RES / "report_numbers.md", "with", len(NUMBERS), "blocks")